# 6.3 — Inventory Aging Dashboard

## 1. Overview

This notebook builds an inventory-aging extract for Tableau, using the same data a planner would pull from SAP transaction **MB52** (stock on hand) or **MMBE** (stock overview).

**Input:** `data/raw/materials_inventory.csv` (registered as `materials`).

**Output:** `data/tableau/inventory_aging.hyper`.

## 2. Load & inspect

In [ ]:
from analytics_bootcamp.duckdb_utils import get_connection, query_df
import pandas as pd

con = get_connection()
df = query_df("SELECT * FROM materials", con)
print(df.shape)
df.head()

In [ ]:
df.dtypes

## 3. Clean

In [ ]:
df = df.copy()
df["LAST_MOVEMENT_DATE"] = pd.to_datetime(df["LAST_MOVEMENT_DATE"], errors="coerce")
df["LABST"] = pd.to_numeric(df["LABST"], errors="coerce").fillna(0)
df["STPRS"] = pd.to_numeric(df["STPRS"], errors="coerce").fillna(0)
df.isna().sum()

## 4. Analytical layer

Aging buckets, dead-stock flag, and a slow-moving-by-plant summary.

In [ ]:
aging = con.sql("""
    SELECT
        MATNR,
        WERKS,
        LGORT,
        LABST,
        LAST_MOVEMENT_DATE,
        DATEDIFF('day',
            TRY_CAST(LAST_MOVEMENT_DATE AS DATE),
            CURRENT_DATE
        ) AS days_since_movement,
        CASE
            WHEN DATEDIFF('day', TRY_CAST(LAST_MOVEMENT_DATE AS DATE), CURRENT_DATE) <= 30  THEN '0-30'
            WHEN DATEDIFF('day', TRY_CAST(LAST_MOVEMENT_DATE AS DATE), CURRENT_DATE) <= 60  THEN '31-60'
            WHEN DATEDIFF('day', TRY_CAST(LAST_MOVEMENT_DATE AS DATE), CURRENT_DATE) <= 90  THEN '61-90'
            ELSE '90+'
        END AS aging_bucket
    FROM materials
    ORDER BY days_since_movement DESC
""").df()
aging.head()

In [ ]:
bucket_counts = (
    aging.groupby("aging_bucket")
         .agg(materials=("MATNR", "count"), units=("LABST", "sum"))
         .reindex(["0-30", "31-60", "61-90", "90+"])
)
bucket_counts

In [ ]:
aging["dead_stock"] = aging["days_since_movement"] > 180
aging["dead_stock"].value_counts()

In [ ]:
slow_by_plant = (
    aging[aging["days_since_movement"] > 90]
    .groupby("WERKS")
    .agg(slow_materials=("MATNR", "count"), slow_units=("LABST", "sum"))
    .sort_values("slow_materials", ascending=False)
)
slow_by_plant

## 5. Visualize

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(bucket_counts.index, bucket_counts["materials"])
ax.set_title("Material Count by Aging Bucket")
ax.set_xlabel("Days Since Last Movement")
ax.set_ylabel("# Materials")
plt.tight_layout()
plt.show()

## 6. Export to .hyper

In [ ]:
from analytics_bootcamp.tableau import to_hyper

to_hyper(aging, "data/tableau/inventory_aging.hyper", "Inventory_Aging")

## 7. Tableau tips

**Filters that planners actually use**
- `WERKS` (plant) — top-level filter, usually the first thing they pick.
- `LGORT` (storage location) — second-level filter inside a plant.
- `aging_bucket` — quick toggle between "only dead stock" and the full picture.

**Drill-down path**
1. Start with a stacked bar of material count (or stock value) by `aging_bucket`, split by `WERKS`.
2. Click a bucket → action filter into a detail table of `MATNR`, `LAST_MOVEMENT_DATE`, `LABST`, `days_since_movement`.
3. Add a calculated field `stock_value = LABST * STPRS` (compute in SQL or in Tableau) and switch the bar chart to value when needed.

**Highlight dead stock** — color the `90+` bucket red and put a KPI tile at the top showing `# dead-stock SKUs`.